# 04 — NodeXL export for manuscript figures

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 04 — Export NodeXL files for revised Figure 3 and Figure 4

This notebook **does not rerun community detection**.

It uses the frozen outputs from the latest corrected Notebook 01:

- Social layer: the existing CNM community assignments from `user_community_df.csv`
- Semantic layer: the existing Leiden topic assignments from `semantic_node_topic_map_revision.csv`

It only rebuilds/exports the edge and vertex tables needed for NodeXL.

### Revised manuscript targets

**Figure 3 — Social interaction network**
- Full LCC: 6,789 users
- Full LCC: 9,914 directed user-to-user edges
- 71 CNM communities
- Figure view: top 20 communities

**Figure 4 — Semantic similarity network**
- 3,405 semantic nodes
- 19,763 undirected semantic edges at cosine similarity ≥ 0.70
- 188 Leiden communities
- Figure view: top 5 semantic communities

### Final top-5 semantic labels

1. Caregiving and Family Support
2. Dementia Suspicions and Uncertainty
3. Family Loss and Grief
4. Alzheimer's Research on Diagnosis and Treatment
5. Dementia Risk and Public Health

**Important:** In NodeXL, do not ask NodeXL to detect communities again. Use the exported `Group` / community membership.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# EDIT ONLY THIS PATH IF YOUR revision output folder moved.
# ============================================================
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
MAIN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "main_pipeline_no_brain_fog_final_v1"

EXPORT_DIR = MAIN_OUTPUT_DIR / "nodexl_exports_final_v1"

SEMANTIC_THRESHOLD = 0.70
SEMANTIC_BLOCK_SIZE = 512
TOP_N_SOCIAL = 20
TOP_N_SEMANTIC = 5

STRICT_EXPECTED_COUNTS = True

EXPECTED = {
    "social_lcc_nodes": 6789,
    "social_lcc_edges": 9914,
    "social_communities": 71,
    "social_top20_nodes": 5438,
    "semantic_nodes": 3405,
    "semantic_edges": 19763,
    "semantic_communities": 188,
}

TOPIC_LABELS = {
    0: "Caregiving and Family Support",
    1: "Suspicions and Uncertainty",
    2: "Family Loss and Grief",
    3: "Research on Diagnosis and Treatment",
    4: "Risk and Public Health",
}

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Clear only prior CSV/TXT exports in this dedicated subfolder.
for p in EXPORT_DIR.iterdir():
    if p.is_file() and p.suffix.lower() in {".csv", ".txt"}:
        p.unlink()

print("MAIN_OUTPUT_DIR:", MAIN_OUTPUT_DIR)
print("EXPORT_DIR     :", EXPORT_DIR)


In [ ]:
# ============================================================
# Required Notebook 01 outputs
# ============================================================
PATHS = {
    "posts": MAIN_OUTPUT_DIR / "filtered_posts_df_2_corrected_propagated.parquet",
    "social_communities": MAIN_OUTPUT_DIR / "user_community_df.csv",
    "centrality": MAIN_OUTPUT_DIR / "centrality_df.csv",
    "embeddings": MAIN_OUTPUT_DIR / "embeddings_revision.npy",
    "semantic_map": MAIN_OUTPUT_DIR / "semantic_node_topic_map_revision.csv",
    "semantic_edge_count": MAIN_OUTPUT_DIR / "semantic_edge_count_revision.txt",
}

required = [
    PATHS["posts"],
    PATHS["social_communities"],
    PATHS["centrality"],
    PATHS["embeddings"],
    PATHS["semantic_map"],
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required Notebook 01 outputs are missing:\n\n"
        + "\n".join(missing)
        + "\n\nCheck MAIN_OUTPUT_DIR above."
    )

for key, path in PATHS.items():
    status = "FOUND" if path.exists() else "optional / not found"
    print(f"{key:22s} -> {status}")

In [ ]:
# ============================================================
# Load frozen Notebook 01 outputs
# ============================================================
posts = pd.read_parquet(PATHS["posts"])
user_community_df = pd.read_csv(PATHS["social_communities"])
centrality_df = pd.read_csv(PATHS["centrality"])
semantic_map = pd.read_csv(PATHS["semantic_map"])
embeddings = np.load(PATHS["embeddings"])

posts["user_id"] = pd.to_numeric(posts["user_id"], errors="coerce").astype("Int64")
posts["partner_id"] = pd.to_numeric(posts["partner_id"], errors="coerce").astype("Int64")

user_community_df["user_id"] = pd.to_numeric(
    user_community_df["user_id"], errors="raise"
).astype("int64")
user_community_df["community"] = pd.to_numeric(
    user_community_df["community"], errors="raise"
).astype(int)

centrality_df["user_id"] = pd.to_numeric(
    centrality_df["user_id"], errors="raise"
).astype("int64")

semantic_map["semantic_node_id"] = pd.to_numeric(
    semantic_map["semantic_node_id"], errors="raise"
).astype(int)
semantic_map["topic"] = pd.to_numeric(
    semantic_map["topic"], errors="raise"
).astype(int)

print("Loaded:")
print(f"  propagated analytic rows : {len(posts):,}")
print(f"  social LCC users         : {len(user_community_df):,}")
print(f"  centrality rows          : {len(centrality_df):,}")
print(f"  embedding rows           : {embeddings.shape[0]:,}")
print(f"  semantic mapped nodes    : {len(semantic_map):,}")

## A. Social interaction network export

The social graph is reconstructed from the expanded interaction rows, but **community assignments are not recomputed**.

A NodeXL edge is one unique directed user-to-user dyad.

Two weight fields are exported:

- `Weight`: reproduces the current Notebook 01 `DiGraph` edge-weight logic.
- `Total Interactions`: total reply/repost/quote records on that dyad.

For Figure 3, use the **TOP20** files if you want the simplest direct import.

In [ ]:
# ============================================================
# Social edges — rebuild the LCC edge list
# ============================================================
social_rows = posts.loc[
    posts["partner_id"].notna() & posts["interaction_type"].notna(),
    ["user_id", "partner_id", "interaction_type"]
].copy()

social_rows = social_rows.dropna(subset=["user_id", "partner_id"])
social_rows["source"] = social_rows["user_id"].astype("int64")
social_rows["target"] = social_rows["partner_id"].astype("int64")
social_rows["interaction_type"] = social_rows["interaction_type"].astype(str)

# user_community_df contains exactly the Notebook 01 LCC users.
lcc_users = set(user_community_df["user_id"].astype("int64"))

social_rows_lcc = social_rows.loc[
    social_rows["source"].isin(lcc_users)
    & social_rows["target"].isin(lcc_users)
].copy()

# Counts by directed dyad and interaction type.
type_counts = (
    social_rows_lcc
    .groupby(["source", "target", "interaction_type"], sort=True)
    .size()
    .unstack(fill_value=0)
)

for t in ["quote", "reply", "repost"]:
    if t not in type_counts.columns:
        type_counts[t] = 0

type_counts = type_counts[["quote", "reply", "repost"]].reset_index()

# Reproduce the current Notebook 01 NetworkX DiGraph attribute overwrite logic.
# Since grouped edge types are sorted quote -> reply -> repost:
# repost takes precedence if present, then reply, then quote.
def analysis_edge_type(row):
    if row["repost"] > 0:
        return "repost"
    if row["reply"] > 0:
        return "reply"
    return "quote"

type_counts["Analysis Edge Type"] = type_counts.apply(analysis_edge_type, axis=1)
type_counts["Weight"] = type_counts.apply(
    lambda r: int(r[r["Analysis Edge Type"]]),
    axis=1
)

type_counts["Total Interactions"] = (
    type_counts["quote"] + type_counts["reply"] + type_counts["repost"]
).astype(int)

comm_map = user_community_df.set_index("user_id")["community"]

type_counts["Source Community"] = type_counts["source"].map(comm_map).astype(int)
type_counts["Target Community"] = type_counts["target"].map(comm_map).astype(int)
type_counts["Community Relation"] = np.where(
    type_counts["Source Community"].eq(type_counts["Target Community"]),
    "Within community",
    "Across communities",
)

# Prefix IDs so Excel/NodeXL cannot silently round long numeric IDs.
type_counts["Vertex 1"] = "U_" + type_counts["source"].astype(str)
type_counts["Vertex 2"] = "U_" + type_counts["target"].astype(str)

social_edges_full = type_counts[[
    "Vertex 1",
    "Vertex 2",
    "Weight",
    "Total Interactions",
    "Analysis Edge Type",
    "quote",
    "reply",
    "repost",
    "Source Community",
    "Target Community",
    "Community Relation",
    "source",
    "target",
]].rename(columns={
    "quote": "Quote Count",
    "reply": "Reply Count",
    "repost": "Repost Count",
    "source": "Source User ID",
    "target": "Target User ID",
})

print(f"Social LCC unique directed edges: {len(social_edges_full):,}")

In [ ]:
# ============================================================
# Social vertices and group tables
# ============================================================
social_sizes = (
    user_community_df["community"]
    .value_counts()
    .rename_axis("Community")
    .reset_index(name="Community Size")
)

social_sizes["Community Rank"] = (
    social_sizes["Community Size"]
    .rank(method="first", ascending=False)
    .astype(int)
)
social_sizes = social_sizes.sort_values(
    ["Community Rank", "Community"]
).reset_index(drop=True)

top20_social_communities = set(
    social_sizes.head(TOP_N_SOCIAL)["Community"].astype(int)
)

social_vertices_full = (
    user_community_df
    .rename(columns={"user_id": "User ID", "community": "Community"})
    .merge(
        social_sizes,
        on="Community",
        how="left",
        validate="many_to_one",
    )
    .merge(
        centrality_df.rename(columns={"user_id": "User ID"}),
        on="User ID",
        how="left",
        validate="one_to_one",
    )
)

social_vertices_full["Vertex"] = "U_" + social_vertices_full["User ID"].astype(str)
social_vertices_full["Group"] = (
    "Social Community " + social_vertices_full["Community"].astype(str)
)
social_vertices_full["Top 20"] = social_vertices_full["Community"].isin(
    top20_social_communities
)

social_vertices_full = social_vertices_full[[
    "Vertex",
    "Group",
    "Top 20",
    "Community",
    "Community Size",
    "Community Rank",
    "Degree",
    "In-degree",
    "Out-degree",
    "Betweenness",
    "Closeness",
    "Eigenvector",
    "Constraint",
    "User ID",
]].sort_values(
    ["Community Rank", "Community", "Vertex"]
).reset_index(drop=True)

social_groups_full = social_sizes.copy()
social_groups_full["Group"] = (
    "Social Community " + social_groups_full["Community"].astype(str)
)
social_groups_full["Visibility"] = np.where(
    social_groups_full["Community"].isin(top20_social_communities),
    "Show",
    "Hide",
)
social_groups_full = social_groups_full[[
    "Group",
    "Community",
    "Community Size",
    "Community Rank",
    "Visibility",
]]

social_group_membership_full = social_vertices_full[["Group", "Vertex"]].copy()

# Direct-use Figure 3 subset.
social_vertices_top20 = social_vertices_full.loc[
    social_vertices_full["Top 20"]
].copy()

top20_vertices = set(social_vertices_top20["Vertex"])

social_edges_top20 = social_edges_full.loc[
    social_edges_full["Vertex 1"].isin(top20_vertices)
    & social_edges_full["Vertex 2"].isin(top20_vertices)
].copy()

print(f"Social communities             : {len(social_sizes):,}")
print(f"Top-20 social users            : {len(social_vertices_top20):,}")
print(f"Edges among top-20 communities : {len(social_edges_top20):,}")

if STRICT_EXPECTED_COUNTS:
    assert len(social_vertices_full) == EXPECTED["social_lcc_nodes"]
    assert len(social_edges_full) == EXPECTED["social_lcc_edges"]
    assert len(social_sizes) == EXPECTED["social_communities"]
    assert len(social_vertices_top20) == EXPECTED["social_top20_nodes"]

print("PASS: social network counts match corrected Notebook 01.")

In [ ]:
# ============================================================
# Save social NodeXL tables
# ============================================================
social_edges_full.to_csv(
    EXPORT_DIR / "Figure3_social_edges_FULL_LCC.csv",
    index=False,
)
social_vertices_full.to_csv(
    EXPORT_DIR / "Figure3_social_vertices_FULL_LCC.csv",
    index=False,
)
social_groups_full.to_csv(
    EXPORT_DIR / "Figure3_social_groups_FULL_LCC.csv",
    index=False,
)
social_group_membership_full.to_csv(
    EXPORT_DIR / "Figure3_social_group_membership_FULL_LCC.csv",
    index=False,
)

social_edges_top20.to_csv(
    EXPORT_DIR / "Figure3_social_edges_TOP20.csv",
    index=False,
)
social_vertices_top20.to_csv(
    EXPORT_DIR / "Figure3_social_vertices_TOP20.csv",
    index=False,
)

print("Saved Figure 3 social-network export files.")

## B. Semantic similarity network export

This section loads the already-computed normalized SBERT embeddings and reconstructs only the **cosine ≥ 0.70 edge list**.

It **does not rerun SBERT** and **does not rerun Leiden**.

The frozen topic/community assignment comes from `semantic_node_topic_map_revision.csv`.

For Figure 4, use the **TOP5** files for the simplest direct import.

In [ ]:
# ============================================================
# Reconstruct semantic threshold edges
# ============================================================
def build_semantic_edges(emb, threshold=0.70, block_size=512):
    """
    Reconstruct the same undirected cosine-threshold graph used in Notebook 01.
    Embeddings are already L2-normalized, so dot product = cosine similarity.
    """
    edges = []
    n_docs = emb.shape[0]

    for start in range(0, n_docs, block_size):
        stop = min(start + block_size, n_docs)
        sims = emb[start:stop] @ emb.T

        for local_i in range(stop - start):
            i = start + local_i
            if i + 1 >= n_docs:
                continue

            row = sims[local_i]
            js = np.flatnonzero(row[i + 1:] >= threshold) + (i + 1)

            if len(js):
                edges.extend(
                    (int(i), int(j), float(row[j]))
                    for j in js
                )

        print(f"  processed {stop:,}/{n_docs:,} embeddings", end="\r")

    print()

    return pd.DataFrame(
        edges,
        columns=["Semantic Node 1", "Semantic Node 2", "Similarity"],
    )

semantic_edges = build_semantic_edges(
    embeddings,
    threshold=SEMANTIC_THRESHOLD,
    block_size=SEMANTIC_BLOCK_SIZE,
)

print(
    f"Semantic edges at threshold {SEMANTIC_THRESHOLD:.2f}: "
    f"{len(semantic_edges):,}"
)

if PATHS["semantic_edge_count"].exists():
    expected_from_file = int(PATHS["semantic_edge_count"].read_text().strip())
    assert len(semantic_edges) == expected_from_file, (
        f"Rebuilt semantic edges={len(semantic_edges):,}; "
        f"saved Notebook 01 edge count={expected_from_file:,}"
    )
    print("PASS: semantic edge count matches semantic_edge_count_revision.txt.")

In [ ]:
# ============================================================
# Semantic vertices and group tables
# ============================================================
semantic_nodes_present = set(
    semantic_edges["Semantic Node 1"].astype(int)
).union(
    set(semantic_edges["Semantic Node 2"].astype(int))
)

mapped_nodes = set(semantic_map["semantic_node_id"].astype(int))

assert semantic_nodes_present == mapped_nodes, (
    f"Semantic endpoint set and frozen topic-map node set differ: "
    f"edges={len(semantic_nodes_present):,}, map={len(mapped_nodes):,}"
)

semantic_sizes = (
    semantic_map["topic"]
    .value_counts()
    .rename_axis("Topic")
    .reset_index(name="Topic Size")
)

semantic_sizes["Topic Rank"] = (
    semantic_sizes["Topic Size"]
    .rank(method="first", ascending=False)
    .astype(int)
)
semantic_sizes = semantic_sizes.sort_values(
    ["Topic Rank", "Topic"]
).reset_index(drop=True)

top5_topics = set(
    semantic_sizes.head(TOP_N_SEMANTIC)["Topic"].astype(int)
)

semantic_vertices_full = semantic_map.rename(columns={
    "semantic_node_id": "Semantic Node ID",
    "original_index": "Original Index",
    "topic": "Topic",
}).merge(
    semantic_sizes,
    on="Topic",
    how="left",
    validate="many_to_one",
)

semantic_vertices_full["Vertex"] = (
    "S_" + semantic_vertices_full["Semantic Node ID"].astype(str)
)
semantic_vertices_full["Display Topic"] = semantic_vertices_full["Topic"] + 1
semantic_vertices_full["Topic Label"] = (
    semantic_vertices_full["Topic"]
    .map(TOPIC_LABELS)
    .fillna("Other semantic community")
)

semantic_vertices_full["Group"] = np.where(
    semantic_vertices_full["Topic"].isin(TOPIC_LABELS.keys()),
    "Topic "
    + semantic_vertices_full["Display Topic"].astype(str)
    + " - "
    + semantic_vertices_full["Topic Label"],
    "Semantic Community " + semantic_vertices_full["Topic"].astype(str),
)

semantic_vertices_full["Top 5"] = semantic_vertices_full["Topic"].isin(
    top5_topics
)

# Optional text preview for inspecting vertices in NodeXL.
if "original_index" in posts.columns:
    rep_text = posts[[
        "original_index",
        "text",
        "text_processed",
    ]].drop_duplicates(subset=["original_index"]).copy()

    semantic_vertices_full = semantic_vertices_full.merge(
        rep_text.rename(columns={"original_index": "Original Index"}),
        on="Original Index",
        how="left",
        validate="one_to_one",
    )

    semantic_vertices_full["Text Preview"] = (
        semantic_vertices_full["text"]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.slice(0, 180)
    )
else:
    semantic_vertices_full["Text Preview"] = ""

semantic_vertices_full = semantic_vertices_full[[
    "Vertex",
    "Group",
    "Top 5",
    "Topic",
    "Display Topic",
    "Topic Label",
    "Topic Size",
    "Topic Rank",
    "Semantic Node ID",
    "Original Index",
    "Text Preview",
]].sort_values(
    ["Topic Rank", "Topic", "Semantic Node ID"]
).reset_index(drop=True)

topic_map = semantic_vertices_full.set_index("Semantic Node ID")["Topic"]

semantic_edges_full = semantic_edges.copy()
semantic_edges_full["Topic 1"] = (
    semantic_edges_full["Semantic Node 1"].map(topic_map).astype(int)
)
semantic_edges_full["Topic 2"] = (
    semantic_edges_full["Semantic Node 2"].map(topic_map).astype(int)
)
semantic_edges_full["Within Same Topic"] = (
    semantic_edges_full["Topic 1"].eq(semantic_edges_full["Topic 2"])
)
semantic_edges_full["Vertex 1"] = (
    "S_" + semantic_edges_full["Semantic Node 1"].astype(str)
)
semantic_edges_full["Vertex 2"] = (
    "S_" + semantic_edges_full["Semantic Node 2"].astype(str)
)
semantic_edges_full["Weight"] = semantic_edges_full["Similarity"]

semantic_edges_full = semantic_edges_full[[
    "Vertex 1",
    "Vertex 2",
    "Weight",
    "Similarity",
    "Topic 1",
    "Topic 2",
    "Within Same Topic",
    "Semantic Node 1",
    "Semantic Node 2",
]]

semantic_groups_full = semantic_sizes.copy()
semantic_groups_full["Display Topic"] = semantic_groups_full["Topic"] + 1
semantic_groups_full["Topic Label"] = (
    semantic_groups_full["Topic"]
    .map(TOPIC_LABELS)
    .fillna("Other semantic community")
)

semantic_groups_full["Group"] = np.where(
    semantic_groups_full["Topic"].isin(TOPIC_LABELS.keys()),
    "Topic "
    + semantic_groups_full["Display Topic"].astype(str)
    + " - "
    + semantic_groups_full["Topic Label"],
    "Semantic Community " + semantic_groups_full["Topic"].astype(str),
)

semantic_groups_full["Visibility"] = np.where(
    semantic_groups_full["Topic"].isin(top5_topics),
    "Show",
    "Hide",
)

semantic_groups_full = semantic_groups_full[[
    "Group",
    "Topic",
    "Display Topic",
    "Topic Label",
    "Topic Size",
    "Topic Rank",
    "Visibility",
]]

semantic_group_membership_full = semantic_vertices_full[["Group", "Vertex"]].copy()

# Direct-use Figure 4 subset.
semantic_vertices_top5 = semantic_vertices_full.loc[
    semantic_vertices_full["Top 5"]
].copy()

top5_sem_vertices = set(semantic_vertices_top5["Vertex"])

semantic_edges_top5 = semantic_edges_full.loc[
    semantic_edges_full["Vertex 1"].isin(top5_sem_vertices)
    & semantic_edges_full["Vertex 2"].isin(top5_sem_vertices)
].copy()

print(f"Semantic nodes           : {len(semantic_vertices_full):,}")
print(f"Semantic communities     : {len(semantic_sizes):,}")
print(f"Top-5 semantic nodes     : {len(semantic_vertices_top5):,}")
print(f"Edges among top-5 topics : {len(semantic_edges_top5):,}")

if STRICT_EXPECTED_COUNTS:
    assert len(semantic_vertices_full) == EXPECTED["semantic_nodes"]
    assert len(semantic_edges_full) == EXPECTED["semantic_edges"]
    assert len(semantic_sizes) == EXPECTED["semantic_communities"]

print("PASS: semantic network counts match corrected Notebook 01.")

In [ ]:
# ============================================================
# Save semantic NodeXL tables
# ============================================================
semantic_edges_full.to_csv(
    EXPORT_DIR / "Figure4_semantic_edges_FULL.csv",
    index=False,
)
semantic_vertices_full.to_csv(
    EXPORT_DIR / "Figure4_semantic_vertices_FULL.csv",
    index=False,
)
semantic_groups_full.to_csv(
    EXPORT_DIR / "Figure4_semantic_groups_FULL.csv",
    index=False,
)
semantic_group_membership_full.to_csv(
    EXPORT_DIR / "Figure4_semantic_group_membership_FULL.csv",
    index=False,
)

semantic_edges_top5.to_csv(
    EXPORT_DIR / "Figure4_semantic_edges_TOP5.csv",
    index=False,
)
semantic_vertices_top5.to_csv(
    EXPORT_DIR / "Figure4_semantic_vertices_TOP5.csv",
    index=False,
)

print("Saved Figure 4 semantic-network export files.")

In [ ]:
# ============================================================
# Validation manifest and NodeXL import instructions
# ============================================================
validation = pd.DataFrame([
    ["Social LCC nodes", len(social_vertices_full), EXPECTED["social_lcc_nodes"]],
    ["Social LCC edges", len(social_edges_full), EXPECTED["social_lcc_edges"]],
    ["Social communities", len(social_sizes), EXPECTED["social_communities"]],
    ["Social top-20 nodes", len(social_vertices_top20), EXPECTED["social_top20_nodes"]],
    ["Semantic nodes", len(semantic_vertices_full), EXPECTED["semantic_nodes"]],
    ["Semantic edges", len(semantic_edges_full), EXPECTED["semantic_edges"]],
    ["Semantic communities", len(semantic_sizes), EXPECTED["semantic_communities"]],
], columns=["Metric", "Actual", "Expected"])

validation["Match"] = validation["Actual"].eq(validation["Expected"])

validation.to_csv(
    EXPORT_DIR / "NodeXL_export_validation_summary.csv",
    index=False,
)

readme_lines = [
    "NODEXL EXPORT FOR REVISED FIGURES 3 AND 4",
    "=========================================",
    "",
    f"Generated from: {MAIN_OUTPUT_DIR}",
    "",
    "FIGURE 3 — SOCIAL INTERACTION NETWORK",
    "-------------------------------------",
    "Simplest files to use:",
    "  Figure3_social_edges_TOP20.csv",
    "  Figure3_social_vertices_TOP20.csv",
    "",
    "NodeXL settings:",
    "  - Graph direction: DIRECTED",
    "  - Import 'Vertex 1' and 'Vertex 2' as edge endpoints.",
    "  - Use 'Group' as the frozen CNM community label.",
    "  - Do NOT run NodeXL community detection.",
    "  - Use 'Weight' for exact consistency with Notebook 01 edge attributes.",
    "  - 'Total Interactions' is also provided as a descriptive strength field.",
    "",
    f"Top 20 social communities contain {len(social_vertices_top20):,} users.",
    "",
    "FIGURE 4 — SEMANTIC SIMILARITY NETWORK",
    "--------------------------------------",
    "Simplest files to use:",
    "  Figure4_semantic_edges_TOP5.csv",
    "  Figure4_semantic_vertices_TOP5.csv",
    "",
    "NodeXL settings:",
    "  - Graph direction: UNDIRECTED",
    "  - Import 'Vertex 1' and 'Vertex 2' as edge endpoints.",
    "  - Use 'Similarity' or 'Weight' for edge strength.",
    "  - Use 'Group' as the frozen Leiden topic label.",
    "  - Do NOT run NodeXL community detection.",
    "",
    f"Semantic threshold: cosine similarity >= {SEMANTIC_THRESHOLD:.2f}",
    "",
    "Final five labels:",
    f"  Topic 1: {TOPIC_LABELS[0]}",
    f"  Topic 2: {TOPIC_LABELS[1]}",
    f"  Topic 3: {TOPIC_LABELS[2]}",
    f"  Topic 4: {TOPIC_LABELS[3]}",
    f"  Topic 5: {TOPIC_LABELS[4]}",
    "",
    "IMPORTANT",
    "---------",
    "The exported group labels come from corrected Notebook 01.",
    "NodeXL is used only for visualization, not for community detection.",
]

(EXPORT_DIR / "README_NodeXL_import.txt").write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8",
)

print(validation.to_string(index=False))
print()
print("All files saved to:")
print(EXPORT_DIR)

if validation["Match"].all():
    print("\nPASS: all frozen-run validation counts match.")
else:
    print("\nWARNING: one or more frozen-run counts do not match.")

## What to import into NodeXL

For the revised figures, you normally only need four files:

### Figure 3
- `Figure3_social_edges_TOP20.csv`
- `Figure3_social_vertices_TOP20.csv`

### Figure 4
- `Figure4_semantic_edges_TOP5.csv`
- `Figure4_semantic_vertices_TOP5.csv`

The `FULL` and group-membership files are retained for reproducibility and if you want to reproduce the previous workflow where all communities were loaded and non-target groups were hidden.

**Do not rerun community detection in NodeXL.**

## C. V3 one-step NodeXL import — recommended

V3 fixes the Figure 3 problem caused by exporting only Top-20 edges.

For **Figure 3**, V3 keeps the **entire social LCC** in NodeXL:

- 6,789 LCC users
- 9,914 directed LCC edges
- all 71 frozen CNM communities

Then NodeXL can display only the largest 20 communities, matching the logic of the old Figure 3 much more closely.

Use:

`Figure3_social_ONE_STEP_IMPORT_FULL_LCC.csv`

The file carries each endpoint's frozen `Social Community`, `Community Rank`, and `Top 20` flag as vertex properties. No XLOOKUP, manual matching, or metadata-carrier self-loops are needed for Figure 3.

After import:

1. `Groups → Group by Vertex Attribute → Social Community`
2. **Do not run community detection**
3. Show the 20 communities listed in `Figure3_social_TOP20_groups.csv`
4. Hide the other 51 communities
5. Use `Lay out each group in its own box → Packed rectangles`

Figure 4 keeps the Top-5 semantic one-step export.

In [ ]:
# ============================================================
# V3 ONE-STEP NODEXL IMPORT FILES
# Figure 3: FULL social LCC, all 71 frozen CNM communities
# Figure 4: Top-5 semantic network
# ============================================================

import csv

def write_csv_with_duplicate_headers(path, headers, rows):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(rows)


# ------------------------------------------------------------
# FIGURE 3 — FULL LCC SOCIAL NETWORK
# ------------------------------------------------------------
social_vertex_meta = (
    social_vertices_full[
        ["Vertex", "Group", "Community Rank", "Top 20"]
    ]
    .set_index("Vertex")
    .to_dict(orient="index")
)

social_full_rows = []

for _, r in social_edges_full.iterrows():
    v1 = r["Vertex 1"]
    v2 = r["Vertex 2"]
    m1 = social_vertex_meta[v1]
    m2 = social_vertex_meta[v2]

    social_full_rows.append([
        v1,
        v2,
        r["Weight"],
        r["Total Interactions"],
        r["Analysis Edge Type"],
        m1["Group"],
        m2["Group"],
        int(m1["Community Rank"]),
        int(m2["Community Rank"]),
        bool(m1["Top 20"]),
        bool(m2["Top 20"]),
    ])

# Every LCC vertex should already occur on at least one real LCC edge.
edge_endpoints = set(social_edges_full["Vertex 1"]).union(
    set(social_edges_full["Vertex 2"])
)
all_lcc_vertices = set(social_vertices_full["Vertex"])

missing_lcc_vertices = sorted(all_lcc_vertices - edge_endpoints)

assert len(missing_lcc_vertices) == 0, (
    f"Unexpected LCC vertices missing from full edge list: "
    f"{len(missing_lcc_vertices)}"
)

social_full_path = (
    EXPORT_DIR / "Figure3_social_ONE_STEP_IMPORT_FULL_LCC.csv"
)

write_csv_with_duplicate_headers(
    social_full_path,
    [
        "Vertex 1",
        "Vertex 2",
        "Weight",
        "Total Interactions",
        "Analysis Edge Type",
        "Social Community",
        "Social Community",
        "Community Rank",
        "Community Rank",
        "Top 20",
        "Top 20",
    ],
    social_full_rows,
)

# Exact Top-20 group list, sorted by size/rank.
top20_groups_export = (
    social_groups_full.loc[
        social_groups_full["Community Rank"] <= TOP_N_SOCIAL,
        ["Group", "Community", "Community Size", "Community Rank"]
    ]
    .sort_values("Community Rank")
    .reset_index(drop=True)
)

top20_groups_export.to_csv(
    EXPORT_DIR / "Figure3_social_TOP20_groups.csv",
    index=False,
)

# Full group manifest with recommended visibility.
all_groups_v3 = social_groups_full.copy()
all_groups_v3["Recommended Visibility"] = np.where(
    all_groups_v3["Community Rank"] <= TOP_N_SOCIAL,
    "Show",
    "Hide",
)

all_groups_v3.to_csv(
    EXPORT_DIR / "Figure3_social_ALL71_groups_with_visibility.csv",
    index=False,
)

print("Figure 3 V3 full-LCC export")
print(f"  nodes             : {len(all_lcc_vertices):,}")
print(f"  edges             : {len(social_edges_full):,}")
print(f"  communities       : {len(social_groups_full):,}")
print(f"  top-20 communities: {len(top20_groups_export):,}")
print(f"  file              : {social_full_path.name}")

if STRICT_EXPECTED_COUNTS:
    assert len(all_lcc_vertices) == EXPECTED["social_lcc_nodes"]
    assert len(social_edges_full) == EXPECTED["social_lcc_edges"]
    assert len(social_groups_full) == EXPECTED["social_communities"]


# ------------------------------------------------------------
# FIGURE 4 — TOP-5 SEMANTIC NETWORK
# ------------------------------------------------------------
semantic_vertex_group = (
    semantic_vertices_top5
    .set_index("Vertex")["Topic Label"]
    .to_dict()
)

semantic_rows = []

for _, r in semantic_edges_top5.iterrows():
    v1 = r["Vertex 1"]
    v2 = r["Vertex 2"]
    semantic_rows.append([
        v1,
        v2,
        r["Similarity"],
        semantic_vertex_group[v1],
        semantic_vertex_group[v2],
    ])

semantic_edge_endpoints = set(
    semantic_edges_top5["Vertex 1"]
).union(
    set(semantic_edges_top5["Vertex 2"])
)

all_top5_sem_vertices = set(semantic_vertices_top5["Vertex"])
missing_semantic_vertices = sorted(
    all_top5_sem_vertices - semantic_edge_endpoints
)

# Keep isolated displayed semantic vertices if any.
for v in missing_semantic_vertices:
    g = semantic_vertex_group[v]
    semantic_rows.append([v, v, 0.0, g, g])

semantic_path = (
    EXPORT_DIR / "Figure4_semantic_ONE_STEP_IMPORT_TOP5.csv"
)

write_csv_with_duplicate_headers(
    semantic_path,
    [
        "Vertex 1",
        "Vertex 2",
        "Similarity",
        "Semantic Topic",
        "Semantic Topic",
    ],
    semantic_rows,
)

print()
print("Figure 4 Top-5 export")
print(f"  real edges            : {len(semantic_edges_top5):,}")
print(f"  metadata carrier rows : {len(missing_semantic_vertices):,}")
print(f"  displayed vertices    : {len(all_top5_sem_vertices):,}")
print(f"  file                  : {semantic_path.name}")


# ------------------------------------------------------------
# README
# ------------------------------------------------------------
readme = [
    "V3 NODEXL IMPORT — RECOMMENDED",
    "===============================",
    "",
    "FIGURE 3",
    "--------",
    "Use ONLY Figure3_social_ONE_STEP_IMPORT_FULL_LCC.csv",
    "",
    "Import > From Open Workbook",
    "Check: Columns have headers",
    "",
    "Map:",
    "  Vertex 1 -> Edge column / Vertex 1 endpoint",
    "  Vertex 2 -> Edge column / Vertex 2 endpoint",
    "  Weight -> Edge column",
    "  Total Interactions -> Edge column",
    "  Analysis Edge Type -> Edge column",
    "  first Social Community -> Vertex 1 Property ONLY",
    "  second Social Community -> Vertex 2 Property ONLY",
    "  first Community Rank -> Vertex 1 Property ONLY",
    "  second Community Rank -> Vertex 2 Property ONLY",
    "  first Top 20 -> Vertex 1 Property ONLY",
    "  second Top 20 -> Vertex 2 Property ONLY",
    "",
    "Then:",
    "  Groups > Group by Vertex Attribute > Social Community",
    "  Do NOT run community detection",
    "  Show groups listed in Figure3_social_TOP20_groups.csv",
    "  Hide the remaining 51 groups",
    "  Layout each group in its own box > Packed rectangles",
    "",
    "Expected full social network:",
    f"  Nodes: {EXPECTED['social_lcc_nodes']}",
    f"  Edges: {EXPECTED['social_lcc_edges']}",
    f"  Communities: {EXPECTED['social_communities']}",
]

(EXPORT_DIR / "README_V3_NodeXL_IMPORT.txt").write_text(
    "\n".join(readme) + "\n",
    encoding="utf-8",
)

print()
print("PASS: V3 NodeXL export files created.")

## D. V4 easy Figure 3 import — no manual searching for 20 groups

This export is designed only to make the NodeXL visualization workflow fast and safe.

It keeps the **full 6,789-node / 9,914-edge LCC**, but adds a visualization-only vertex property called `Display Group`:

- Top-20 communities become rank-prefixed groups such as  
  `R01 | Social Community 1`, `R02 | Social Community 16`, ..., `R20 | ...`
- All remaining 51 communities are assigned to one visualization group:  
  `R99 | Other Communities`

The original frozen CNM community ID, community rank, and Top-20 flag are still carried as separate vertex properties, so no analytic information is lost.

### NodeXL workflow

1. Import `Figure3_social_ONE_STEP_IMPORT_V4_EASY.csv`
2. Group by vertex attribute **Display Group**
3. NodeXL creates only **21 visualization groups**
4. Hide only **one group**: `R99 | Other Communities`
5. Keep the other 20 visible
6. Use Group-in-a-Box → Packed rectangles

This avoids manually searching through 71 groups.

In [ ]:
# ============================================================
# V4 EASY NODEXL EXPORT
# Full social LCC, but visualization grouping = Top20 + one Other group.
# ============================================================

import csv

def write_dup_header_csv(path, headers, rows):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(headers)
        w.writerows(rows)

# Build rank-prefixed display group names.
v4_meta = social_vertices_full[
    ["Vertex", "Group", "Community Rank", "Top 20"]
].copy()

v4_meta["Display Group"] = np.where(
    v4_meta["Top 20"],
    "R"
    + v4_meta["Community Rank"].astype(int).astype(str).str.zfill(2)
    + " | "
    + v4_meta["Group"],
    "R99 | Other Communities",
)

meta = v4_meta.set_index("Vertex").to_dict(orient="index")

rows = []

for _, r in social_edges_full.iterrows():
    v1 = r["Vertex 1"]
    v2 = r["Vertex 2"]
    m1 = meta[v1]
    m2 = meta[v2]

    rows.append([
        v1,
        v2,
        r["Weight"],
        r["Total Interactions"],
        r["Analysis Edge Type"],
        m1["Display Group"],
        m2["Display Group"],
        m1["Group"],
        m2["Group"],
        int(m1["Community Rank"]),
        int(m2["Community Rank"]),
        bool(m1["Top 20"]),
        bool(m2["Top 20"]),
    ])

v4_path = EXPORT_DIR / "Figure3_social_ONE_STEP_IMPORT_V4_EASY.csv"

write_dup_header_csv(
    v4_path,
    [
        "Vertex 1",
        "Vertex 2",
        "Weight",
        "Total Interactions",
        "Analysis Edge Type",
        "Display Group",
        "Display Group",
        "Social Community",
        "Social Community",
        "Community Rank",
        "Community Rank",
        "Top 20",
        "Top 20",
    ],
    rows,
)

# A tiny 21-group manifest in the exact intended visualization order.
display_groups = (
    v4_meta[["Display Group", "Community Rank", "Top 20"]]
    .drop_duplicates()
    .sort_values(["Top 20", "Community Rank"], ascending=[False, True])
)

# Collapse the 51 non-top20 rows into one manifest row.
display_groups = pd.concat([
    display_groups.loc[display_groups["Top 20"]].copy(),
    pd.DataFrame([{
        "Display Group": "R99 | Other Communities",
        "Community Rank": 99,
        "Top 20": False,
    }])
], ignore_index=True)

display_groups["Recommended Visibility"] = np.where(
    display_groups["Top 20"], "Show", "Hide"
)

display_groups.to_csv(
    EXPORT_DIR / "Figure3_V4_display_groups_21_rows.csv",
    index=False,
)

print("V4 easy Figure 3 export created:")
print(f"  full LCC nodes        : {social_vertices_full['Vertex'].nunique():,}")
print(f"  full LCC edges        : {len(social_edges_full):,}")
print(f"  visualization groups  : {v4_meta['Display Group'].nunique():,}")
print(f"  top-20 groups         : {v4_meta.loc[v4_meta['Top 20'], 'Display Group'].nunique():,}")
print(f"  other group           : 1")
print(f"  file                  : {v4_path.name}")

assert v4_meta["Display Group"].nunique() == 21
assert v4_meta.loc[v4_meta["Top 20"], "Display Group"].nunique() == 20

print("\nPASS: no manual search through 71 groups is needed.")

## E. V5 robust Figure 3 import — guaranteed vertex grouping

V5 fixes the remaining NodeXL issue where some vertices can end up **ungrouped** after import.

Instead of relying on two duplicate `Display Group` columns for Vertex 1 and Vertex 2, V5 uses only **one** vertex-property column:

`Display Group`

It is mapped only to **Vertex 1 Property**.

For every vertex that never appears as `Vertex 1` on a real directed edge, V5 adds one hidden metadata-carrier self-loop. This guarantees that **every LCC vertex appears at least once as Vertex 1** and therefore receives its `Display Group` property.

The carrier rows have `Visibility = 0` (Skip), so they are not shown as real edges.

### NodeXL import mapping

Use:

`Figure3_social_ONE_STEP_IMPORT_V5_ROBUST.csv`

Check **Columns have headers**, then map:

- `Vertex 1` → Edge Column / Vertex 1 endpoint
- `Vertex 2` → Edge Column / Vertex 2 endpoint
- `Weight` → Edge Column
- `Total Interactions` → Edge Column
- `Analysis Edge Type` → Edge Column
- `Visibility` → Edge Column
- `Display Group` → **Vertex 1 Property only**
- `Social Community` → **Vertex 1 Property only**
- `Community Rank` → **Vertex 1 Property only**
- `Top 20` → **Vertex 1 Property only**

Do **not** select any Vertex 2 Property columns.

After import, group by `Display Group`. You should get exactly 21 groups:
R01–R20 plus R99.

In [ ]:
# ============================================================
# V5 ROBUST NODEXL EXPORT
# Guarantee every vertex receives Display Group via Vertex 1.
# ============================================================

import csv

v5_meta = social_vertices_full[
    ["Vertex", "Group", "Community Rank", "Top 20"]
].copy()

v5_meta["Display Group"] = np.where(
    v5_meta["Top 20"],
    "R"
    + v5_meta["Community Rank"].astype(int).astype(str).str.zfill(2)
    + " | "
    + v5_meta["Group"],
    "R99 | Other Communities",
)

meta = v5_meta.set_index("Vertex").to_dict(orient="index")

# Real edge rows: metadata belongs to Vertex 1 only.
rows = []
for _, r in social_edges_full.iterrows():
    v1 = r["Vertex 1"]
    v2 = r["Vertex 2"]
    m1 = meta[v1]

    rows.append([
        v1,
        v2,
        r["Weight"],
        r["Total Interactions"],
        r["Analysis Edge Type"],
        1,  # real edge = Show
        m1["Display Group"],
        m1["Group"],
        int(m1["Community Rank"]),
        bool(m1["Top 20"]),
    ])

# Vertices that never appear as Vertex 1 need one hidden metadata row.
real_sources = set(social_edges_full["Vertex 1"])
all_vertices = set(social_vertices_full["Vertex"])
needs_carrier = sorted(all_vertices - real_sources)

for v in needs_carrier:
    m = meta[v]
    rows.append([
        v,
        v,
        0,
        0,
        "metadata_carrier",
        0,  # Skip hidden self-loop
        m["Display Group"],
        m["Group"],
        int(m["Community Rank"]),
        bool(m["Top 20"]),
    ])

v5_path = EXPORT_DIR / "Figure3_social_ONE_STEP_IMPORT_V5_ROBUST.csv"

with open(v5_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "Vertex 1",
        "Vertex 2",
        "Weight",
        "Total Interactions",
        "Analysis Edge Type",
        "Visibility",
        "Display Group",
        "Social Community",
        "Community Rank",
        "Top 20",
    ])
    writer.writerows(rows)

# Validation: every vertex appears at least once as Vertex 1.
imported_v1 = {r[0] for r in rows}
assert imported_v1 == all_vertices

# Exact intended display groups.
display_group_counts = (
    v5_meta["Display Group"]
    .value_counts()
    .rename_axis("Display Group")
    .reset_index(name="N Vertices")
)

display_group_counts["Recommended Visibility"] = np.where(
    display_group_counts["Display Group"].eq("R99 | Other Communities"),
    "Skip",
    "Show",
)

display_group_counts = display_group_counts.sort_values("Display Group")
display_group_counts.to_csv(
    EXPORT_DIR / "Figure3_V5_display_group_counts.csv",
    index=False,
)

print("V5 robust Figure 3 export created")
print(f"  real LCC edges            : {len(social_edges_full):,}")
print(f"  real LCC vertices         : {len(all_vertices):,}")
print(f"  vertices already Vertex 1 : {len(real_sources):,}")
print(f"  hidden carrier rows       : {len(needs_carrier):,}")
print(f"  display groups            : {v5_meta['Display Group'].nunique():,}")
print(f"  output                    : {v5_path.name}")

assert len(all_vertices) == EXPECTED["social_lcc_nodes"]
assert len(social_edges_full) == EXPECTED["social_lcc_edges"]
assert v5_meta["Display Group"].nunique() == 21

print("\nPASS: every vertex is guaranteed to receive a Display Group via Vertex 1.")

## F. V6 robust Figure 4 import — Top 5 semantic clusters only

This section fixes the remaining NodeXL grouping issue for Figure 4.

The export contains only vertices belonging to the five largest semantic clusters and only edges whose two endpoints are both in those five clusters.

To make NodeXL grouping reliable, the file carries `Semantic Topic` only as a **Vertex 1 property**. If a Top-5 semantic vertex never appears as Vertex 1 on a real retained edge, V6 adds one hidden metadata-carrier self-loop (`Visibility = 0`) so that every displayed semantic vertex receives its topic label.

Use:

`Figure4_semantic_ONE_STEP_IMPORT_V6_ROBUST_TOP5.csv`

NodeXL import mapping:

- `Vertex 1` → Edge Column / Vertex 1 endpoint
- `Vertex 2` → Edge Column / Vertex 2 endpoint
- `Similarity` → Edge Column
- `Visibility` → Edge Column
- `Semantic Topic` → **Vertex 1 Property only**

Do not mark anything as Vertex 2 Property.

After import:

1. `Groups → Group by Vertex Attribute → Semantic Topic`
2. You should get exactly 5 groups
3. Do not run community detection

In [ ]:
# ============================================================
# V6 ROBUST FIGURE 4 EXPORT
# Top-5 semantic clusters only; every vertex gets topic via Vertex 1.
# ============================================================

import csv

sem_v6_meta = semantic_vertices_top5[
    ["Vertex", "Topic", "Display Topic", "Topic Label"]
].copy()

sem_v6_meta["Semantic Topic"] = (
    "Topic "
    + sem_v6_meta["Display Topic"].astype(int).astype(str)
    + " | "
    + sem_v6_meta["Topic Label"]
)

sem_meta = sem_v6_meta.set_index("Vertex").to_dict(orient="index")

sem_v6_rows = []

for _, r in semantic_edges_top5.iterrows():
    v1 = r["Vertex 1"]
    v2 = r["Vertex 2"]
    m1 = sem_meta[v1]

    sem_v6_rows.append([
        v1,
        v2,
        r["Similarity"],
        1,
        m1["Semantic Topic"],
    ])

real_sem_sources = set(semantic_edges_top5["Vertex 1"])
all_sem_top5_vertices = set(semantic_vertices_top5["Vertex"])
sem_needs_carrier = sorted(all_sem_top5_vertices - real_sem_sources)

for v in sem_needs_carrier:
    m = sem_meta[v]
    sem_v6_rows.append([
        v,
        v,
        0.0,
        0,
        m["Semantic Topic"],
    ])

sem_v6_path = EXPORT_DIR / "Figure4_semantic_ONE_STEP_IMPORT_V6_ROBUST_TOP5.csv"

with open(sem_v6_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "Vertex 1",
        "Vertex 2",
        "Similarity",
        "Visibility",
        "Semantic Topic",
    ])
    writer.writerows(sem_v6_rows)

imported_v1 = {r[0] for r in sem_v6_rows}
assert imported_v1 == all_sem_top5_vertices

topic_counts = (
    sem_v6_meta["Semantic Topic"]
    .value_counts()
    .rename_axis("Semantic Topic")
    .reset_index(name="N Vertices")
    .sort_values("N Vertices", ascending=False)
)

topic_counts.to_csv(
    EXPORT_DIR / "Figure4_V6_top5_topic_counts.csv",
    index=False,
)

print("V6 robust Figure 4 export created")
print(f"  Top-5 semantic vertices : {len(all_sem_top5_vertices):,}")
print(f"  Real retained edges     : {len(semantic_edges_top5):,}")
print(f"  Hidden carrier rows     : {len(sem_needs_carrier):,}")
print(f"  Semantic groups         : {sem_v6_meta['Semantic Topic'].nunique():,}")
print(f"  Output                  : {sem_v6_path.name}")

assert sem_v6_meta["Semantic Topic"].nunique() == 5

print("\nTop-5 topic sizes:")
print(topic_counts.to_string(index=False))
print("\nPASS: Figure 4 import contains only the five intended semantic clusters.")